In [24]:
dir = "/Users/abhinavmohanty/Documents/Python/SMS/SMS/"
account = "Account-Table 1.csv"
plan = "Plan-Table 1.csv"
subs = "Subscription-Table 1.csv"
orders = "Order-Table 1.csv"
pse = "PSE-Table 1.csv"

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master( "local[3]" ) \
    .appName('SMS') \
    .getOrCreate()

In [4]:
print("App Name:", spark.sparkContext.appName)

App Name: SMS


In [36]:
df_pse = spark.read.csv(dir + pse, header=True, inferSchema=True)

In [37]:
df_pse.printSchema()

root
 |-- sub_ref_id: integer (nullable = true)
 |-- user_ref_id: integer (nullable = true)
 |-- plan_code: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- expiry_date: string (nullable = true)
 |-- transaction_date: string (nullable = true)



In [38]:
df_subs = spark.read.csv(dir + subs, header=True, inferSchema=True)

In [39]:
df_subs.printSchema()

root
 |-- sub_ref_id: integer (nullable = true)
 |-- user_ref_id: integer (nullable = true)
 |-- plan_code: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- event: string (nullable = true)
 |-- expiry_date: string (nullable = true)



In [40]:
df_orders = spark.read.csv(dir + orders, header=True, inferSchema=True)

In [41]:
df_orders.printSchema()

root
 |-- order_ref_id: integer (nullable = true)
 |-- sub_ref_id: integer (nullable = true)
 |-- user_ref_id: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- order_code: integer (nullable = true)



In [42]:
df_acct = spark.read.csv(dir + account, header=True, inferSchema=True)

In [43]:
df_acct.printSchema()

root
 |-- account_ref_id: integer (nullable = true)
 |-- user_ref_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- Phone: integer (nullable = true)



In [44]:
df_plan = spark.read.csv(dir + plan, header=True, inferSchema=True)

In [45]:
df_plan.printSchema()

root
 |-- plan_code: integer (nullable = true)
 |-- plan_name: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)



In [48]:
df_orders.join(df_subs, 'sub_ref_id', 'inner') \
    .join(df_acct, 'user_ref_id', 'inner') \
    .join(df_plan, 'plan_code', 'inner') \
    .select('transaction_date', 'event', 'name', 'email', 'plan_name', 'type', 'price', 'expiry_date') \
    .show()

+----------------+-----+----+-----------+---------+-----+-----+-----------+
|transaction_date|event|name|      email|plan_name| type|price|expiry_date|
+----------------+-----+----+-----------+---------+-----+-----+-----------+
|      2025-12-13|  new| abc|abc@xyz.com| ultimate|trail|   50| 2026-01-13|
|      2025-12-13|  new| abc|abc@xyz.com| ultimate|trail|   50| 2026-01-13|
|      2025-12-13|  new| abc|abc@xyz.com| ultimate|trail|    0| 2026-01-13|
|      2025-12-13|  new| abc|abc@xyz.com|  advance| paid|   50| 2026-12-13|
|      2025-12-13|  new| abc|abc@xyz.com|  advance| paid|   50| 2026-12-13|
|      2025-12-13|  new| abc|abc@xyz.com|  advance| paid|    0| 2026-12-13|
|      2025-12-13|renew| abc|abc@xyz.com|  advance| paid|   50| 2027-12-13|
|      2025-12-13|renew| abc|abc@xyz.com|  advance| paid|   50| 2027-12-13|
|      2025-12-13|renew| abc|abc@xyz.com|  advance| paid|    0| 2027-12-13|
|      2025-12-13|  new| def|def@xyz.com|  advance| paid|   50| 2026-01-13|
|      2025-

In [47]:
df_pse.show()

+----------+-----------+---------+-----+-----+-----------+----------------+
|sub_ref_id|user_ref_id|plan_code|price| type|expiry_date|transaction_date|
+----------+-----------+---------+-----+-----+-----------+----------------+
|         1|          1|        1|    0|trail| 2026-01-13|      2025-12-13|
|         1|          1|        2|   50| paid| 2026-12-13|      2025-12-13|
|         1|          1|        2|   50| paid| 2027-12-13|      2025-12-13|
|         2|          2|        2|   50| paid| 2026-12-13|      2025-12-13|
|         2|          2|        2|   50| paid| 2027-12-13|      2025-12-13|
+----------+-----------+---------+-----+-----+-----------+----------------+

